# 3. Export for the app (run when you're happy with notebook 2)
Saves the forest from notebook 2 as `models/scope-forest.json` on your Drive, in the compact form the app reads.

In [ ]:
# Connect your Drive and load the shared code
from google.colab import drive
drive.mount("/content/drive")
import sys, importlib
sys.path.insert(0, "/content/drive/MyDrive/kq-scope-forest")
import common; importlib.reload(common)

In [ ]:
import json, os, gzip
import joblib
import pandas as pd
model = joblib.load(f"{common.MODELS}/forest.joblib")
test = pd.read_csv(f"{common.DATA}/test.csv")
off_col = list(model.classes_).index("off_topic")

In [ ]:
def compact(e):
    t = e.tree_
    leaf = t.children_left == -1
    vals = t.value[:, 0, :]
    p_off = vals[:, off_col] / vals.sum(axis=1)
    return {
        "feature": t.feature.tolist(),
        "threshold": [round(float(x), 5) for x in t.threshold],
        "left": t.children_left.tolist(),
        "right": t.children_right.tolist(),
        "p_off": [round(float(p), 4) if l else 0 for p, l in zip(p_off, leaf)],
    }
samples = test.sample(5, random_state=1)
sample_probs = model.predict_proba(common.embed(samples["text"]))[:, off_col]
export = {
    "encoder": "bge-small-en-v1.5",
    "band": [0.3, 0.7],
    "trees": [compact(e) for e in model.estimators_],
    "checks": [{"text": t, "p_off_topic": round(float(p), 4)} for t, p in zip(samples["text"], sample_probs)],
}
path = f"{common.MODELS}/scope-forest.json"
with open(path, "w") as f:
    json.dump(export, f, separators=(",", ":"))
raw = os.path.getsize(path)
packed = len(gzip.compress(open(path, "rb").read()))
print(len(export["trees"]), "trees,", round(raw / 1e6, 2), "MB raw,", round(packed / 1e6, 2), "MB compressed")